[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmolSISL/IMDb-Deep-Learning-Research/blob/main/notebooks/01_IMDb_Data_Exploration.ipynb)

# Notebook 01 — IMDb Data Preparation and Exploratory Data Analysis

**Project:** Customer Voice Intelligence: A Deep Learning Framework for Review Sentiment Analysis, Cross-Domain Transfer, and Foundation Model Architecture in Streaming and E-Commerce
**Author:** Amol Birajdar · Walsh College · QM 640 Data Analytics Capstone · Mentor: Dr. Rishabh Pandey

This notebook reproduces the **Data Description** and **Analysis** sections of the report:

| Step | Output in report |
|---|---|
| Download and verify the IMDb Large Movie Review Dataset v1.0 (Maas et al., 2011) | Dataset overview |
| Parse ratings from file names, clean HTML, remove duplicates, flag outliers and explicit score mentions | Table 3 (cleaning log) |
| Stratified 70/10/20 train / validation / development-test split (seed 42); official test set held out | Processed file `imdb_clean.csv` |
| Rating, length, vocabulary, discriminative-word and negation analyses | Figures 1–5, Table 4 |
| Dataset preview and folder structure | Figures 9–10 |

It also writes the shared helper module `code/cvi_utils.py` that the later notebooks (02–07) import.

**Run order:** 01 → 02 → 03 → 04 → 05 → 06 → 07. All files are kept in Google Drive under `MyDrive/customer-voice-intelligence/`, so each notebook can be run in a fresh Colab session.

## 0. Setup

In [ ]:
# ---------------------------------------------------------------
# Project setup: mount Google Drive so every notebook shares the same
# data, models and outputs (folder: MyDrive/customer-voice-intelligence)
# ---------------------------------------------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/customer-voice-intelligence"
else:  # running locally
    PROJECT_DIR = os.environ.get("CVI_PROJECT_DIR", os.path.abspath("customer-voice-intelligence"))

for sub in ["data/raw", "data/processed", "outputs/figures", "outputs/tables",
            "outputs/results", "models", "code"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)
sys.path.insert(0, os.path.join(PROJECT_DIR, "code"))
print("Project folder:", PROJECT_DIR)

### 0.1 Shared helper module
The cell below writes `code/cvi_utils.py`, which holds the cleaning rules, tokenizer, metrics and the McNemar test so that every notebook uses **identical** definitions (a requirement for paired comparisons across research questions).

In [ ]:
%%writefile {PROJECT_DIR}/code/cvi_utils.py
"""Shared utilities for the Customer Voice Intelligence capstone (QM 640)."""
import os, re, html, json, random, time
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

PROJECT_DIR = os.environ.get("CVI_PROJECT_DIR_RESOLVED", None)
QUICK = os.environ.get("CVI_QUICK") == "1"   # small-sample smoke test mode
SEED = 42


def set_project(path):
    global PROJECT_DIR
    PROJECT_DIR = path


def P(*parts):
    """Absolute path inside the project folder."""
    return os.path.join(PROJECT_DIR, *parts)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


# ----------------------------------------------------------------- text cleaning
HTML_RE = re.compile(r"<[^>]+>|&[a-zA-Z]+;|&#\d+;")
TAG_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"\s+")
# explicit scores such as "7/10", "7 / 10", "8.5/10", "3 out of 10"
SCORE_RE = re.compile(r"\b(?:10|\d(?:\.\d)?)\s*(?:/|out\s+of)\s*10\b", re.IGNORECASE)
SCORE_TOKEN = " scoremask "


def has_html(text):
    return bool(HTML_RE.search(text))


def clean_text(text):
    """Replace HTML tags by spaces, decode entities, collapse whitespace."""
    text = TAG_RE.sub(" ", str(text))
    text = html.unescape(text)
    return WS_RE.sub(" ", text).strip()


def has_score(text):
    return bool(SCORE_RE.search(text))


def mask_scores(text):
    """Replace explicit numeric scores by a neutral placeholder (sensitivity test)."""
    return SCORE_RE.sub(SCORE_TOKEN, text)


# ----------------------------------------------------------------- tokenisation
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[!?]")


def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())


PAD, UNK = 0, 1


def build_vocab(texts, size=20000):
    from collections import Counter
    cnt = Counter()
    for t in texts:
        cnt.update(tokenize(t))
    itos = ["<pad>", "<unk>"] + [w for w, _ in cnt.most_common(size)]
    return {w: i for i, w in enumerate(itos)}


def encode(texts, vocab, max_len=400):
    """Token ids, truncated at the end and zero-padded; returns ids and lengths."""
    X = np.zeros((len(texts), max_len), dtype=np.int64)
    lens = np.zeros(len(texts), dtype=np.int64)
    for i, t in enumerate(texts):
        ids = [vocab.get(w, UNK) for w in tokenize(t)][:max_len]
        if not ids:
            ids = [UNK]
        X[i, :len(ids)] = ids
        lens[i] = len(ids)
    return X, lens


# ----------------------------------------------------------------- evaluation
def classification_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)
    out = {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1_pos": f1_score(y_true, pred, pos_label=1, zero_division=0),
        "f1_neg": f1_score(y_true, pred, pos_label=0, zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
    }
    try:
        out["roc_auc"] = roc_auc_score(y_true, prob)
    except ValueError:
        out["roc_auc"] = np.nan
    tp = int(((pred == 1) & (y_true == 1)).sum()); tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum()); fn = int(((pred == 0) & (y_true == 1)).sum())
    out.update(tp=tp, tn=tn, fp=fp, fn=fn, n=len(y_true))
    return out


def accuracy_ci(acc, n, z=1.96):
    half = z * np.sqrt(acc * (1 - acc) / n)
    return acc - half, acc + half


def mcnemar(y_true, pred_a, pred_b):
    """Continuity-corrected McNemar test (exact binomial when b + c < 25).
    b = A right & B wrong, c = A wrong & B right."""
    y_true, pred_a, pred_b = map(lambda a: np.asarray(a).astype(int), (y_true, pred_a, pred_b))
    a_ok, b_ok = pred_a == y_true, pred_b == y_true
    b = int((a_ok & ~b_ok).sum()); c = int((~a_ok & b_ok).sum())
    if b + c == 0:
        return dict(b=0, c=0, disagree=0.0, chi2=0.0, p=1.0, test="none")
    if b + c < 25:
        p = stats.binomtest(min(b, c), b + c, 0.5).pvalue
        chi2, test = np.nan, "exact"
    else:
        chi2 = (abs(b - c) - 1) ** 2 / (b + c)
        p = stats.chi2.sf(chi2, df=1); test = "chi2_cc"
    return dict(b=b, c=c, disagree=(b + c) / len(y_true), chi2=chi2, p=p, test=test)


def fmt_p(p):
    return "< .001" if p < 0.001 else f"{p:.3f}".lstrip("0")


# ----------------------------------------------------------------- I/O helpers
def save_table(df, name, index=False):
    path = P("outputs", "tables", name if name.endswith(".csv") else name + ".csv")
    df.to_csv(path, index=index)
    return path


def save_fig(fig, name, dpi=200):
    path = P("outputs", "figures", name if name.endswith(".png") else name + ".png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    return path


def load_imdb():
    return pd.read_csv(P("data", "processed", "imdb_clean.csv"), keep_default_na=False)


def load_amazon():
    return pd.read_csv(P("data", "processed", "amazon_electronics_20k.csv"), keep_default_na=False)


class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter(); return self
    def __exit__(self, *a):
        self.seconds = time.perf_counter() - self.t0


def append_csv(row_or_df, path):
    """Append results to a CSV (used to resume long experiments after a Colab disconnect)."""
    df = row_or_df if isinstance(row_or_df, pd.DataFrame) else pd.DataFrame([row_or_df])
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)


# ----------------------------------------------------------------- plotting style
def plot_style():
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 11,
        "axes.titleweight": "bold", "legend.frameon": False,
    })

POS_COLOR, NEG_COLOR = "#2a7ab9", "#d1495b"
MODEL_COLORS = {"TF-IDF + LogReg": "#4c566a", "Feed-forward NN": "#2a7ab9",
                "1D-CNN": "#e09f3e", "BiLSTM": "#9c4f96", "DistilBERT": "#2e933c",
                "LLM": "#d1495b"}

In [ ]:
import importlib, cvi_utils as U
importlib.reload(U)
U.set_project(PROJECT_DIR)

import re, tarfile, urllib.request, shutil, json
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

U.plot_style()
pd.set_option("display.max_colwidth", 120)
U.set_seed(42)

## 1. Data acquisition

The IMDb Large Movie Review Dataset v1.0 is downloaded from the **original Stanford source** (Maas et al., 2011). The Stanford archive is used rather than the Hugging Face mirror because the review **ratings (1–10) are only encoded in the original file names**, and the ratings are needed for the label check and the rating analysis.

The archive (~84 MB) is stored in `data/raw/` on Drive and extracted to the fast local Colab disk. The 50,000 unlabeled reviews (`unsup/`) are not used and are skipped.

In [ ]:
IMDB_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
RAW_TAR = U.P("data", "raw", "aclImdb_v1.tar.gz")
EXTRACT_DIR = os.environ.get("CVI_IMDB_DIR") or ("/content/aclImdb_extract" if IN_COLAB else U.P("data", "raw", "extract"))
IMDB_ROOT = os.path.join(EXTRACT_DIR, "aclImdb")

if not os.path.isdir(os.path.join(IMDB_ROOT, "train", "pos")):
    if not os.path.exists(RAW_TAR):
        print("Downloading", IMDB_URL)
        urllib.request.urlretrieve(IMDB_URL, RAW_TAR)
    print("Archive size: %.1f MB" % (os.path.getsize(RAW_TAR) / 1e6))
    with tarfile.open(RAW_TAR, "r:gz") as tar:
        members = [m for m in tar.getmembers() if "/unsup/" not in m.name and not m.name.endswith("unsupBow.feat")]
        tar.extractall(EXTRACT_DIR, members=members)
print("IMDb folder:", IMDB_ROOT)
for split in ["train", "test"]:
    for pol in ["pos", "neg"]:
        d = os.path.join(IMDB_ROOT, split, pol)
        print(f"  {split}/{pol}: {len([f for f in os.listdir(d) if f.endswith('.txt')]):,} files")

### 1.1 Parse reviews and recover ratings
Each review is stored as `<id>_<rating>.txt`. The rating is parsed from the file name, and the label is taken from the folder (`pos` = 1, `neg` = 0).

In [ ]:
def read_split(root, split):
    rows = []
    for pol, lab in [("pos", 1), ("neg", 0)]:
        folder = os.path.join(root, split, pol)
        for fn in sorted(os.listdir(folder)):
            if not fn.endswith(".txt"):
                continue
            fid, rating = fn[:-4].split("_")
            with open(os.path.join(folder, fn), encoding="utf-8") as f:
                txt = f.read()
            rows.append(dict(review_id=f"imdb_{split}_{pol}_{fid}", raw_text=txt,
                             rating=int(rating), label=lab, source_split=split))
    return pd.DataFrame(rows)

raw_train = read_split(IMDB_ROOT, "train")
raw_test = read_split(IMDB_ROOT, "test")
print("Raw training reviews:", len(raw_train), "| positive:", raw_train.label.sum(), "| negative:", (raw_train.label == 0).sum())
print("Raw official test reviews:", len(raw_test))
raw_train.head(3)

## 2. Data cleaning

Each check in Table 3 is applied in order. Every number in the cleaning log is computed here rather than typed in.

In [ ]:
log = []   # rows of Table 3
N = len(raw_train)

def pct(k, n=N):
    return f"{k:,} ({100 * k / n:.1f}%)" if k / n >= 0.01 or k == 0 else f"{k:,} ({100 * k / n:.2f}%)"

# 2.1 Missing values ---------------------------------------------------------
missing = int((raw_train.raw_text.str.strip() == "").sum() + raw_train[["raw_text", "rating", "label"]].isna().any(axis=1).sum())
log.append(["Missing values", "review_text, rating, label", "Null and empty-string check",
            f"None required ({missing} of {N:,})" if missing == 0 else f"Dropped {pct(missing)}", "Confirms complete records"])

# 2.2 Label-rating conflicts -------------------------------------------------
def conflicts(df):
    return int((((df.label == 1) & (df.rating < 7)) | ((df.label == 0) & (df.rating > 4))).sum())
n_conf = conflicts(raw_train)
log.append(["Label-rating conflicts", "rating, label", "Rule check: positive >= 7, negative <= 4",
            f"None required ({n_conf} conflicts)" if n_conf == 0 else f"Dropped {n_conf}", "Validates target definition"])

# 2.3 HTML artefacts ---------------------------------------------------------
for df in (raw_train, raw_test):
    df["had_html"] = df.raw_text.map(U.has_html)
    df["review_text"] = df.raw_text.map(U.clean_text)
n_html = int(raw_train.had_html.sum())
log.append(["HTML artefacts", "review_text", "Regular expression for tags and entities",
            f"Tags replaced by spaces, entities decoded: {pct(n_html)}", "Prevents markup tokens in vocabulary"])

# drop missing / conflicts (none expected)
train = raw_train[(raw_train.review_text != "") &
                  ~(((raw_train.label == 1) & (raw_train.rating < 7)) | ((raw_train.label == 0) & (raw_train.rating > 4)))].copy()

# 2.4 Exact duplicates (after cleaning) ---------------------------------------
dup_mask = train.duplicated("review_text", keep="first")
dup_label_conflicts = int((train.groupby("review_text").label.nunique() > 1).sum())
n_dup = int(dup_mask.sum())
train = train[~dup_mask].copy()
log.append(["Duplicates", "review_text", "Exact match after cleaning",
            f"First kept, {n_dup} removed ({100 * n_dup / N:.2f}%); {dup_label_conflicts} label conflicts", "Avoids train-test leakage"])

# 2.5 Length outliers (IQR rule on whitespace tokens) ---------------------------
train["token_count"] = train.review_text.str.split().str.len()
q1, q3 = train.token_count.quantile([0.25, 0.75])
FENCE = int(np.floor(q3 + 1.5 * (q3 - q1)))
train["is_long_outlier"] = (train.token_count > FENCE).astype(int)
n_out = int(train.is_long_outlier.sum())
log.append(["Outliers", "token_count", f"IQR rule, fence = Q3 + 1.5 IQR = {FENCE} tokens",
            f"Flagged, retained: {pct(n_out, len(train))}; truncated to 400 tokens in sequence models", "Long reviews are valid data"])

# 2.6 Explicit score mentions (possible label leakage) --------------------------
train["has_score"] = train.review_text.map(U.has_score).astype(int)
n_score = int(train.has_score.sum())
log.append(["Explicit scores in text", "review_text", 'Regular expression for "x/10" and "x out of 10"',
            f"Flagged for sensitivity test: {pct(n_score, len(train))}", "Possible label leakage (see Notebook 03)"])

table3 = pd.DataFrame(log, columns=["Issue", "Variables Affected", "Detection Method", "Treatment Applied", "Rationale"])
U.save_table(table3, "table3_cleaning_log")
print(f"Reviews retained after cleaning: {len(train):,} of {N:,}")
table3

### 2.1 Official test set
The official 25,000-review test set receives the **same cleaning**. It is not used for any modelling decision until the final evaluation in Notebook 03. Internal duplicates are removed, and any test review whose text also appears in the training partition is dropped, so that no test review was seen in training.

In [ ]:
test = raw_test[raw_test.review_text != ""].copy()
n_test_raw = len(test)
test_dup = test.duplicated("review_text", keep="first")
test = test[~test_dup].copy()
leak = test.review_text.isin(set(train.review_text))
test = test[~leak].copy()
test["token_count"] = test.review_text.str.split().str.len()
test["is_long_outlier"] = (test.token_count > FENCE).astype(int)   # fence from training data only
test["has_score"] = test.review_text.map(U.has_score).astype(int)
print(f"Official test: {n_test_raw:,} raw -> {len(test):,} kept "
      f"({int(test_dup.sum())} internal duplicates, {int(leak.sum())} duplicates of a training review removed)")
print("Label-rating conflicts in test:", conflicts(test))

## 3. Train / validation / development-test split
The cleaned training partition is split **70 / 10 / 20**, stratified by label with seed 42. The development-test partition is used for interim checks. The official test set is labelled `test` and is reserved for the final evaluation.

In [ ]:
rest, dev_test = train_test_split(train, test_size=0.20, stratify=train.label, random_state=42)
tr, val = train_test_split(rest, test_size=0.125, stratify=rest.label, random_state=42)   # 0.125 x 0.8 = 10%
tr = tr.assign(split="train"); val = val.assign(split="validation"); dev_test = dev_test.assign(split="dev_test")
test = test.assign(split="test")

COLS = ["review_id", "review_text", "rating", "label", "domain", "token_count", "is_long_outlier", "has_score", "split"]
imdb = pd.concat([tr, val, dev_test, test], ignore_index=True).assign(domain="movie")[COLS]
imdb.to_csv(U.P("data", "processed", "imdb_clean.csv"), index=False)

split_summary = (imdb.groupby("split", sort=False)
                 .agg(reviews=("label", "size"), positive_share=("label", "mean"),
                      median_tokens=("token_count", "median"))
                 .reindex(["train", "validation", "dev_test", "test"]))
split_summary["positive_share"] = (100 * split_summary.positive_share).round(1).astype(str) + "%"
U.save_table(split_summary, "imdb_split_summary", index=True)
print("Saved data/processed/imdb_clean.csv with", len(imdb), "rows and", len(COLS), "columns")
split_summary

## 4. Exploratory data analysis
The rating, length and marker analyses use the **whole cleaned training partition** (train + validation + development test). The vocabulary statistics and discriminative-word scores use the **training split only**, so that no information from the validation or development-test partitions influences modelling choices.

In [ ]:
eda = imdb[imdb.split != "test"].copy()          # cleaned training partition
trn = imdb[imdb.split == "train"].copy()           # vocabulary statistics only
print(f"EDA set: {len(eda):,} reviews | training split: {len(trn):,}")

### Figure 1 — Distribution of reviewer ratings

In [ ]:
counts = eda.rating.value_counts().reindex(range(1, 11), fill_value=0)
fig, ax = plt.subplots(figsize=(7, 3.6))
colors = [U.NEG_COLOR if r <= 4 else U.POS_COLOR for r in counts.index]
bars = ax.bar(counts.index, counts.values, color=colors)
for r, v in counts.items():
    ax.text(r, v + 40, f"{v:,}" if v else "excluded", ha="center", fontsize=8, color="0.3" if v else "0.5")
ax.set_xticks(range(1, 11)); ax.set_xlabel("Reviewer rating (1–10)"); ax.set_ylabel("Number of reviews")
ax.set_title("Figure 1. Rating distribution, cleaned IMDb training partition")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=U.NEG_COLOR, label="Negative (≤ 4)"), Patch(color=U.POS_COLOR, label="Positive (≥ 7)")], loc="upper center")
U.save_fig(fig, "fig1_rating_distribution"); plt.show()

ext = counts[1] + counts[10]
print(f"Extreme ratings (1 or 10): {ext:,} = {100 * ext / len(eda):.1f}% of reviews")
print(f"Class balance: positive {eda.label.sum():,} vs negative {(eda.label == 0).sum():,}")
print("Intermediate ratings range:", counts[[2, 3, 4, 7, 8, 9]].min(), "to", counts[[2, 3, 4, 7, 8, 9]].max())

### Figure 2 — Review length by sentiment class and by rating

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.4, 1]})
clip = eda.token_count.quantile(0.99)
bins = np.linspace(0, clip, 60)
for lab, col, name in [(0, U.NEG_COLOR, "Negative"), (1, U.POS_COLOR, "Positive")]:
    s = eda.loc[eda.label == lab, "token_count"]
    axes[0].hist(s.clip(upper=clip), bins=bins, alpha=0.55, color=col, label=f"{name} (median {int(s.median())})")
axes[0].set_xlabel("Tokens per review (clipped at 99th percentile)"); axes[0].set_ylabel("Reviews")
axes[0].set_title("(a) Length distribution by class"); axes[0].legend()

med = eda.groupby("rating").token_count.median()
axes[1].bar(med.index, med.values, color=[U.NEG_COLOR if r <= 4 else U.POS_COLOR for r in med.index])
for r, v in med.items():
    axes[1].text(r, v + 3, int(v), ha="center", fontsize=8)
axes[1].set_xticks(med.index); axes[1].set_xlabel("Rating"); axes[1].set_ylabel("Median tokens")
axes[1].set_title("(b) Median length by rating")
fig.suptitle("Figure 2. Review length", fontweight="bold", y=1.02)
U.save_fig(fig, "fig2_length_by_class"); plt.show()

tc = eda.token_count
r_pb = np.corrcoef(tc, eda.label)[0, 1]
print(f"Median {tc.median():.0f}, mean {tc.mean():.0f}, 90th pct {tc.quantile(.9):.0f}, min {tc.min()}, max {tc.max()} tokens")
print(f"Point-biserial correlation between length and label: {r_pb:.3f}")
print("Median length by rating:", med.to_dict())

### Figure 3 — Vocabulary coverage and sequence-length coverage
These two curves set the vocabulary size (20,000) and the maximum sequence length (400) used by the CNN and BiLSTM. The token counts here use the same tokenizer as the models (`cvi_utils.tokenize`).

In [ ]:
tok_counts = Counter()
seq_lens = []
for t in trn.review_text:
    toks = U.tokenize(t)
    tok_counts.update(toks)
    seq_lens.append(len(toks))
seq_lens = np.array(seq_lens)
freqs = np.array(sorted(tok_counts.values(), reverse=True))
cum = np.cumsum(freqs) / freqs.sum()
V = len(freqs)
cov = {k: cum[min(k, V) - 1] for k in [5000, 10000, 20000, 30000]}
hapax = (freqs == 1).mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(np.arange(1, V + 1), 100 * cum, color=U.POS_COLOR)
axes[0].set_xscale("log")
for k, dy in [(5000, -22), (20000, -44)]:
    axes[0].axvline(k, ls="--", color="0.5", lw=0.8)
    axes[0].annotate(f"{k // 1000}k words: {100 * cov[k]:.1f}%", (k, 100 * cov[k]), xytext=(-125, dy), textcoords="offset points", fontsize=8,
                     arrowprops=dict(arrowstyle="-", color="0.5"))
axes[0].set_xlabel("Vocabulary size (most frequent words, log scale)"); axes[0].set_ylabel("% of all tokens covered")
axes[0].set_title("(a) Vocabulary coverage")

L = np.arange(0, np.percentile(seq_lens, 99.5))
frac = [(seq_lens <= l).mean() * 100 for l in L]
axes[1].plot(L, frac, color=U.NEG_COLOR)
for l in [256, 400, 512]:
    f = (seq_lens <= l).mean() * 100
    axes[1].axvline(l, ls="--", color="0.5", lw=0.8)
    axes[1].text(l + 8, f - 9, f"{l}: {f:.0f}%", fontsize=8)
axes[1].set_xlabel("Maximum sequence length (tokens)"); axes[1].set_ylabel("% of reviews not truncated")
axes[1].set_title("(b) Sequence-length coverage")
fig.suptitle("Figure 3. Vocabulary and sequence-length coverage (training split)", fontweight="bold", y=1.02)
U.save_fig(fig, "fig3_vocab_seq_coverage"); plt.show()

print(f"Distinct word types: {V:,}; occurring once: {100 * hapax:.0f}%")
for k, v in cov.items():
    print(f"  top {k:,} words cover {100 * v:.1f}% of tokens")
for l in [256, 400, 512]:
    print(f"  max length {l}: {100 * (seq_lens <= l).mean():.0f}% of reviews fit; {100 * (seq_lens > l).mean():.0f}% truncated")

### Figure 4 — Most discriminative words (weighted log-odds with an informative Dirichlet prior)
Following Monroe et al. (2008), for each word *w* the log-odds ratio between the positive and negative classes is smoothed with a prior equal to the word's frequency in the whole training split, and divided by its estimated standard deviation to give a z-score. |z| > 1.96 marks a statistically distinctive word at the 5% level. English stop words are excluded from the display, and a word must appear in at least 20 training reviews.

In [ ]:
cv = CountVectorizer(tokenizer=U.tokenize, lowercase=False, token_pattern=None, min_df=20)
Xc = cv.fit_transform(trn.review_text)
words = np.array(cv.get_feature_names_out())
y_pos = np.asarray(Xc[(trn.label == 1).values].sum(0)).ravel().astype(float)
y_neg = np.asarray(Xc[(trn.label == 0).values].sum(0)).ravel().astype(float)
alpha = y_pos + y_neg                      # informative prior = background frequency
a0, n_pos, n_neg = alpha.sum(), y_pos.sum(), y_neg.sum()
delta = (np.log((y_pos + alpha) / (n_pos + a0 - y_pos - alpha))
         - np.log((y_neg + alpha) / (n_neg + a0 - y_neg - alpha)))
z = delta / np.sqrt(1 / (y_pos + alpha) + 1 / (y_neg + alpha))
lo = pd.DataFrame({"word": words, "z": z, "count": alpha.astype(int)})
lo = lo[~lo.word.isin(ENGLISH_STOP_WORDS) & lo.word.str.match(r"^[a-z][a-z']+$")]
top_pos = lo.nlargest(15, "z"); top_neg = lo.nsmallest(15, "z")
U.save_table(pd.concat([top_pos, top_neg]), "fig4_logodds_top_words")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].barh(top_neg.word[::-1], top_neg.z[::-1], color=U.NEG_COLOR); axes[0].set_title("(a) Negative reviews")
axes[1].barh(top_pos.word[::-1], top_pos.z[::-1], color=U.POS_COLOR); axes[1].set_title("(b) Positive reviews")
for ax in axes:
    ax.set_xlabel("z-score (weighted log-odds)"); ax.axvline(0, color="0.3", lw=0.6)
fig.suptitle("Figure 4. Most discriminative words by sentiment", fontweight="bold", y=1.01)
fig.tight_layout(); U.save_fig(fig, "fig4_discriminative_words"); plt.show()

print("Distinctive words (|z| > 1.96):", int((np.abs(lo.z) > 1.96).sum()), "of", len(lo))
for w in ["movie", "minutes", "film", "plot"]:
    if w in set(lo.word):
        print(f"  '{w}': z = {lo.loc[lo.word == w, 'z'].iloc[0]:.1f}")

### Figure 5 — Negation, contrast and punctuation markers

In [ ]:
NEG_WORDS = {"not", "no", "never", "nothing", "nobody", "none", "neither", "nor", "cannot"}
CONTRAST = {"but", "however", "although", "though", "yet"}

def marker_counts(text):
    toks = U.tokenize(text)
    return pd.Series({
        "tokens": len(toks),
        'Negation ("not"/"n\'t")': sum(t == "not" or t.endswith("n't") for t in toks),
        "Other negation (no, never, ...)": sum(t in NEG_WORDS - {"not"} for t in toks),
        "Contrast (but, however, ...)": sum(t in CONTRAST for t in toks),
        "Question mark": toks.count("?"),
        "Exclamation mark": toks.count("!"),
    })

mk = eda.review_text.apply(marker_counts)
mk["label"] = eda.label.values
tot = mk.groupby("label").sum()
rates = (tot.drop(columns="tokens").div(tot.tokens, axis=0) * 100).T
rates.columns = ["Negative", "Positive"]
rates["Ratio (neg / pos)"] = rates.Negative / rates.Positive
U.save_table(rates.round(3), "fig5_marker_rates", index=True)

fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(len(rates)); w = 0.38
ax.bar(x - w / 2, rates.Negative, w, color=U.NEG_COLOR, label="Negative")
ax.bar(x + w / 2, rates.Positive, w, color=U.POS_COLOR, label="Positive")
for i, (n, p) in enumerate(zip(rates.Negative, rates.Positive)):
    ax.text(i - w / 2, n + 0.02, f"{n:.2f}", ha="center", fontsize=8); ax.text(i + w / 2, p + 0.02, f"{p:.2f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(rates.index, fontsize=8.5)
ax.set_ylabel("Occurrences per 100 tokens"); ax.legend()
ax.set_title("Figure 5. Negation, contrast and punctuation markers by sentiment")
U.save_fig(fig, "fig5_negation_markers"); plt.show()
rates.round(2)

### Table 4 — EDA insight summary
The table text is generated from the numbers computed above, so it stays consistent with the figures if the data are refreshed.

In [ ]:
neg_ratio = rates.loc['Negation ("not"/"n\'t")', "Ratio (neg / pos)"]
table4 = pd.DataFrame([
    ["Figure 1", "Rating distribution", f"Classes balanced ({eda.label.mean():.1%} positive); {100 * ext / len(eda):.0f}% of reviews are 1 or 10",
     "Accuracy is valid; models may be over-confident on extremes", "No resampling; test borderline Amazon ratings in RQ2"],
    ["Figure 2", "Length by class", f"Length carries almost no label signal (r = {r_pb:.3f}); moderate ratings are longest",
     "Long, mixed reviews are the hard cases (RQ1)", "Report long-review accuracy separately"],
    ["Figure 3", "Vocabulary and length coverage", f"20k words cover {100 * cov[20000]:.1f}% of tokens; 400 tokens keep {100 * (seq_lens <= 400).mean():.0f}% of reviews",
     "Controls model size and cost (RQ1)", "Vocabulary 20,000; max length 400"],
    ["Figure 4", "Discriminative words", 'Domain words ("movie", "minutes") are strong negative cues',
     "Signals will not transfer to products (RQ2)", "Quantify domain-shift drop in RQ2"],
    ["Figure 5", "Negation markers", f"Negation {neg_ratio:.1f}x more frequent in negative reviews",
     "Word order matters (RQ1)", "Use bigrams; compare CNN and BiLSTM"],
    ["Table 3", "Cleaning log", f"{n_dup} duplicates; {100 * n_score / len(train):.1f}% of reviews state a score",
     "Data quality and leakage (RQ3, validity)", "De-duplicate; run score-masking sensitivity test"],
], columns=["Figure/Table", "What it Shows", "Key Insight", "Why it Matters (Link to RQ)", "Decision/Next Step"])
U.save_table(table4, "table4_eda_insights")
table4

## 5. Evidence for the report (Figures 9 and 10)

In [ ]:
preview = imdb.head(8).copy()
preview["review_text"] = preview.review_text.str.slice(0, 60) + "…"
fig, ax = plt.subplots(figsize=(13, 2.6)); ax.axis("off")
tbl = ax.table(cellText=preview.values, colLabels=preview.columns, loc="center", cellLoc="left")
tbl.auto_set_font_size(False); tbl.set_fontsize(7); tbl.auto_set_column_width(list(range(len(preview.columns))))
ax.set_title("Figure 9. Preview of the processed IMDb dataset (imdb_clean.csv)", fontweight="bold")
U.save_fig(fig, "fig9_dataset_preview"); plt.show()

def tree(root, max_depth=2, prefix=""):
    lines = []
    entries = sorted(e for e in os.listdir(root) if not e.startswith(".") and e != "extract")
    for i, e in enumerate(entries):
        p = os.path.join(root, e); last = i == len(entries) - 1
        lines.append(prefix + ("└── " if last else "├── ") + e + ("/" if os.path.isdir(p) else ""))
        if os.path.isdir(p) and max_depth > 1:
            lines += tree(p, max_depth - 1, prefix + ("    " if last else "│   "))
    return lines

tree_txt = "customer-voice-intelligence/\n" + "\n".join(tree(PROJECT_DIR, 2))
print(tree_txt)
fig, ax = plt.subplots(figsize=(6, 0.18 * tree_txt.count("\n") + 0.8)); ax.axis("off")
ax.text(0, 1, tree_txt, family="monospace", fontsize=8, va="top")
ax.set_title("Figure 10. Project folder structure", fontweight="bold", loc="left")
U.save_fig(fig, "fig10_folder_structure"); plt.show()

## Summary
* `data/processed/imdb_clean.csv` holds the cleaned IMDb data with the split column (`train`, `validation`, `dev_test`, `test`).
* Table 3, Table 4 and Figures 1–5, 9 and 10 are saved in `outputs/`.

**Next:** run `02_Amazon_Sampling.ipynb` to build the target-domain sample.